<a href="https://colab.research.google.com/github/rayaneghilene/PIDA-M2-Courses/blob/main/Couse_2_language_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Course 2: From word vectors to usable language models - Master II PIDA


**Goal:** run existing models, inspect their outputs, and explain what they can and cannot do.

Open this notebook in Google Colab using **File → Upload notebook**. Run cells from top to bottom with the ▶ button. Change only the text and settings identified in the exercises. Python knowledge is not required today.

We use public or invented text only. Colab runs on Google's cloud: this is NOT local processing on your laptop. The three models below are public and do not require a Hugging Face token. CPU is sufficient for short examples; GPU is optional.

**Today we mostly perform inference:** supplying inputs to already-trained models. Only the tiny Word2Vec demonstration below trains a toy model.


## 1. Install Dependencies

In [1]:
%pip -q install "transformers==4.57.1" "gensim==4.4.0" pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 80.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 47.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 51.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:
import numpy as np
import pandas as pd
import torch
import transformers
from IPython.display import display
from transformers import pipeline, set_seed

DEVICE = 0 if torch.cuda.is_available() else -1
print("Execution device:", "GPU" if DEVICE == 0 else "CPU")
print("Transformers version:", transformers.__version__)

Execution device: GPU
Transformers version: 4.57.1


## 2. Word2Vec: one fixed vector per vocabulary word

Compare:
- "I deposited money at the **bank**."
- "I sat beside the river **bank**."

A standard trained Word2Vec model looks up the same vector for the vocabulary word `bank` in both sentences. It learned that vector from contexts during training, but the lookup does not depend on this new sentence.

**Precise wording:** context-independent/static at inference, not “invariant in every way.” Retraining or selecting another model can change the vector. A sentence representation made by averaging word vectors can also change when the other words change.

The model below is deliberately tiny and trains in seconds. It demonstrates the lookup mechanism, NOT reliable semantic relationships or word analogies.


In [3]:
from gensim.models import Word2Vec

corpus = [
    "i deposited money at the bank",
    "the bank approved a loan",
    "the customer opened a bank account",
    "i sat beside the river bank",
    "the river bank was covered with grass",
    "the boat reached the river bank",
]
training_sentences = [sentence.split() for sentence in corpus] * 30
word2vec = Word2Vec(
    sentences=training_sentences, vector_size=20, window=3,
    min_count=1, workers=1, seed=42, epochs=30, sg=1
)

In [6]:
sentences = [
    "i deposited money at the bank",
    "i sat beside the river bank",
]

# The sentence is shown for context, but Word2Vec receives only the word.
vectors = [word2vec.wv["bank"].copy() for sentence in sentences]
for sentence, vector in zip(sentences, vectors):
    print(sentence)
    print("bank vector (first 6 coordinates):", vector[:6])

print("Exactly the same vector?", np.array_equal(vectors[0], vectors[1]))
print("Similarity (cosine):", float(np.dot(vectors[0], vectors[1]) / (
    np.linalg.norm(vectors[0]) * np.linalg.norm(vectors[1])
)))


i deposited money at the bank
bank vector (first 6 coordinates): [ 0.13753924  0.34179875 -0.18180406  0.5527664   0.10297513  0.27423906]
i sat beside the river bank
bank vector (first 6 coordinates): [ 0.13753924  0.34179875 -0.18180406  0.5527664   0.10297513  0.27423906]
Exactly the same vector? True
Similarity (cosine): 0.9999999403953552


## 3. What is Hugging Face?

Hugging Face is an ecosystem for sharing and using machine-learning resources.

| Resource | What you find there |
|---|---|
| Models | Model repositories: weights, configuration, tokenizer, model card |
| Datasets | Data repositories and descriptions |
| Spaces | Interactive applications/demos; not the same as model repositories |
| Transformers | Python library used here to load and run compatible models |
| pipeline | A convenient interface that combines preprocessing, model inference and output formatting |

### Find a model together

1. Open https://huggingface.co/models
2. Filter by **task**: Token Classification / Fill-Mask / Text Generation.
3. Check the **language**.
4. Open the **model card**: intended use, training data, limitations, evaluation, licence.
5. Inspect **Files and versions** and **Use this model**.
6. Copy the repository ID, for example `dslim/bert-base-NER`.

Downloads/likes are popularity signals, not proof of suitability. A model's scores and benchmark performance do not guarantee success on your corpus. Publicly downloadable does not automatically mean unrestricted licensing.

**Browse these models, but do not change IDs until the guided examples work:**
- NER: https://huggingface.co/dslim/bert-base-NER
- MLM: https://huggingface.co/google-bert/bert-base-uncased
- CLM: https://huggingface.co/distilbert/distilgpt2

Official references: https://huggingface.co/docs/hub/model-cards and https://huggingface.co/docs/transformers/main_classes/pipelines


___

## 4. NER: extract actors, organizations and places

**Application:** help index a corpus of political speeches or news articles by the actors and locations mentioned.

**Predict first:** which spans should receive which labels?

The selected model was fine-tuned on English CoNLL-2003 news data. It recognizes PER (person), ORG (organization), LOC (location), and MISC (miscellaneous). It does not identify every possible entity type. Its training domain and language limit what we should expect.

BERT supplies contextual representations; a token-classification head assigns labels. That head is NOT a Transformer decoder.


In [7]:
ner = pipeline(
    "token-classification",
    model="dslim/bert-base-NER",
    aggregation_strategy="simple",
    device=DEVICE,
)


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/829 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Device set to use cuda:0


In [8]:
# TO DO
# CHANGE THIS TEXT after trying the supplied example.
text = "Emmanuel Macron met Ursula von der Leyen in Paris to discuss European AI policy."


entities = ner(text)

display(pd.DataFrame([
    {"text": text[e["start"]:e["end"]], "label": e["entity_group"],
     "score": float(e["score"]), "start": e["start"], "end": e["end"]}
    for e in entities
], columns=["text", "label", "score", "start", "end"]))


,text,label,score,start,end
0,Emmanuel Macron,PER,0.999468,0,15
1,Ursula von der Leye,PER,0.997642,20,39
2,Paris,LOC,0.999533,44,49
3,European AI,MISC,0.892270,61,72


### Let's apply this to a small corpus

Apply the same procedure to several texts and obtain a table. The code is provided; edit only the list of texts.


In [9]:
texts = [
    "Emmanuel Macron spoke in Paris.",
    "The United Nations held a meeting in New York.",
    "Microsoft announced an investment in France.",
]
rows = []
for document_id, document in enumerate(texts):
    for entity in ner(document):
        rows.append({
            "document_id": document_id,
            "entity": document[entity["start"]:entity["end"]],
            "type": entity["entity_group"],
            "score": float(entity["score"]),
        })
entity_table = pd.DataFrame(rows, columns=["document_id", "entity", "type", "score"])
display(entity_table)
entity_table.to_csv("named_entities.csv", index=False)
print("Saved named_entities.csv in this runtime; download it from Colab's Files panel.")


,document_id,entity,type,score
0,0,Emmanuel Macron,PER,0.999358
1,0,Paris,LOC,0.999655
2,1,United Nations,ORG,0.998813
3,1,New York,LOC,0.999451
4,2,Microsoft,ORG,0.998582
5,2,France,LOC,0.999588


Saved named_entities.csv in this runtime; download it from Colab's Files panel.


## 5. Masked Language Modeling (MLM): predict a missing token using both sides

**Application:** explore how context influences predictions and formulate hypotheses about learned associations. This simple probe is NOT a validated measurement of societal bias by itself.


## Masked Token (word) prediction

In [17]:
from transformers import pipeline

fill_mask = pipeline(
    "fill-mask",
    model="google-bert/bert-base-uncased"
)

MASK = fill_mask.tokenizer.mask_token

Some weights of the model checkpoint at google-bert/bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cuda:0


,candidate,score,completed_text
0,paris,0.052343,the politician travelled to paris to attend th...
1,london,0.042295,the politician travelled to london to attend t...
2,switzerland,0.028058,the politician travelled to switzerland to att...
3,france,0.023393,the politician travelled to france to attend t...
4,moscow,0.023117,the politician travelled to moscow to attend t...


### Tryout some other examples :

In [18]:
# Use one mask per sentence in this exercise.
sentence = f"The politician travelled to {MASK} to attend the summit."
results = fill_mask(sentence, top_k=5)
display(pd.DataFrame([
    {"candidate": r["token_str"], "score": float(r["score"]), "completed_text": r["sequence"]}
    for r in results
]))

,candidate,score,completed_text
0,paris,0.052343,the politician travelled to paris to attend th...
1,london,0.042295,the politician travelled to london to attend t...
2,switzerland,0.028058,the politician travelled to switzerland to att...
3,france,0.023393,the politician travelled to france to attend t...
4,moscow,0.023117,the politician travelled to moscow to attend t...


In [19]:
sentence = f"The nurse said that {MASK} was exhausted."
results = fill_mask(sentence, top_k=5)
display(pd.DataFrame([
    {"candidate": r["token_str"], "score": float(r["score"]), "completed_text": r["sequence"]}
    for r in results
]))

,candidate,score,completed_text
0,she,0.625168,the nurse said that she was exhausted.
1,he,0.169666,the nurse said that he was exhausted.
2,i,0.097382,the nurse said that i was exhausted.
3,everyone,0.005473,the nurse said that everyone was exhausted.
4,it,0.001702,the nurse said that it was exhausted.


In [20]:
sentence = f"The engineer said that {MASK} was exhausted."

results = fill_mask(sentence, top_k=5)
display(pd.DataFrame([
    {"candidate": r["token_str"], "score": float(r["score"]), "completed_text": r["sequence"]}
    for r in results
]))

,candidate,score,completed_text
0,he,0.896105,the engineer said that he was exhausted.
1,she,0.039884,the engineer said that she was exhausted.
2,i,0.008165,the engineer said that i was exhausted.
3,it,0.006618,the engineer said that it was exhausted.
4,everyone,0.001119,the engineer said that everyone was exhausted.


In [24]:
sentence = f"The immigrant was described as {MASK}."

results = fill_mask(sentence, top_k=5)
display(pd.DataFrame([
    {"candidate": r["token_str"], "score": float(r["score"]), "completed_text": r["sequence"]}
    for r in results
]))

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


,candidate,score,completed_text
0,jewish,0.052082,the illegal immigrant was described as jewish.
1,black,0.045282,the illegal immigrant was described as black.
2,female,0.033002,the illegal immigrant was described as female.
3,white,0.030779,the illegal immigrant was described as white.
4,insane,0.029804,the illegal immigrant was described as insane.


### What do you notice ?

___


In [25]:
bank_sentences = [
    "I deposited money at the bank.",
    "I sat beside the river bank.",
]
bert = fill_mask.model.base_model
bert.eval()
contextual_vectors = []

for sentence in bank_sentences:
    inputs = fill_mask.tokenizer(sentence, return_tensors="pt")
    tokens = fill_mask.tokenizer.convert_ids_to_tokens(inputs["input_ids"][0].tolist())
    bank_index = tokens.index("bank")
    bank_id = int(inputs["input_ids"][0, bank_index])
    inputs = {key: value.to(bert.device) for key, value in inputs.items()}
    with torch.no_grad():
        vector = bert(**inputs).last_hidden_state[0, bank_index].cpu().numpy()
    contextual_vectors.append(vector)
    print(sentence)
    print("bank token ID:", bank_id)
    print("Contextual vector (first 6 coordinates):", vector[:6])

first, second = contextual_vectors
print("Exactly equal?", np.array_equal(first, second))
print("Cosine similarity:", float(np.dot(first, second) / (
    np.linalg.norm(first) * np.linalg.norm(second)
)))


I deposited money at the bank.
bank token ID: 2924
Contextual vector (first 6 coordinates): [ 0.6792881  -0.4071214  -0.06882733  0.0782634   0.8408113   0.14632845]
I sat beside the river bank.
bank token ID: 2924
Contextual vector (first 6 coordinates): [ 0.18160188 -0.5836795   0.0525604  -0.22406489 -0.48745686  0.11813012]
Exactly equal? False
Cosine similarity: 0.5411713123321533


## 6. CLM: predict the next token and continue

**Application:** text continuation; more broadly, causal language models underpin many generative systems.

DistilGPT2 is a small, older base causal language model. It is intentionally used to expose continuation, not to showcase the best chatbot performance. It is NOT instruction-tuned: it may continue a question rather than answer it. Output may be false, biased or inappropriate.

MLM: context on both sides → masked-token prediction.
CLM: preceding tokens → next-token prediction → repeat.


In [26]:
generator = pipeline(
    "text-generation", model="distilbert/distilgpt2", device=DEVICE
)


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0


In [30]:
# CHANGE THIS PROMPT after trying the supplied example.
prompt = "Artificial intelligence will change political communication because"
set_seed(42)
outputs = generator(
    prompt, max_new_tokens=40, do_sample=True,
    temperature=0.8, top_p=0.9, num_return_sequences=2,
    return_full_text=False, pad_token_id=generator.tokenizer.eos_token_id,
)
for i, output in enumerate(outputs, 1):
    print(f"\nCONTINUATION {i}:")
    print(prompt + output["generated_text"])



CONTINUATION 1:
Artificial intelligence will change political communication because it will be able to use the same data to communicate with people, and it will be able to use the same data to communicate with people.












CONTINUATION 2:
Artificial intelligence will change political communication because it will be able to make decisions on the basis of intelligence,” he said.
























### CLM exercise

1. Run the same cell with another seed: e.g. `set_seed(7)`.
2. Keep the prompt and seed fixed, then compare temperatures 0.5 and 1.2.
3. Give it a question. Does it always answer?


---

## 7. What actually happened inside pipeline?

| Layer | What happens | Our example |
|---|---|---|
| Task | Select the requested operation | token-classification / fill-mask / text-generation |
| Model ID | Select a particular trained model | dslim/bert-base-NER |
| Preprocessing | Turn text into token IDs and other tensors | tokenizer |
| Inference | Run the model without training its weights | forward pass / generation |
| Postprocessing | Convert outputs into readable results | labels, candidate tokens, generated text |

**It is not automatically an API call to a hosted model:** here weights are downloaded and computation runs in the Colab runtime. Colab itself is hosted infrastructure, not your own laptop.

You can change the text without retraining the model. You can choose another model, but it must support the task and fit the available resources. Do not paste access tokens into shared notebooks or enable `trust_remote_code=True` without understanding and trusting the code.


## 8. Final recall and research judgement

| Tool | Input | Output | Typical training signal |
|---|---|---|---|
| Word2Vec | Word/context pairs during training; a vocabulary word at lookup | Static word vector | Automatically constructed context-prediction examples |
| NER model | Text | Entity spans and labels | Supervised entity labels for fine-tuning |
| MLM | Text with a mask | Candidate missing tokens | Self-supervised masked tokens |
| CLM | Prefix | Next tokens / continuation | Self-supervised next tokens; possible later post-training |

A model may have several training stages. These task names are not four mutually exclusive learning paradigms.

For one proposed research use, answer:
1. Which tool would you choose and why?
2. What input would you provide?
3. What output could become a useful table or annotation?
4. How would you evaluate errors on your corpus?
5. What decision must remain with the researcher?




### Model/reference links
- Word2Vec tutorial: https://radimrehurek.com/gensim/auto_examples/tutorials/run_word2vec.html
- Pipelines: https://huggingface.co/docs/transformers/main_classes/pipelines
- Model cards: https://huggingface.co/docs/hub/model-cards
- NER card: https://huggingface.co/dslim/bert-base-NER
- BERT card: https://huggingface.co/google-bert/bert-base-uncased
- DistilGPT2 card: https://huggingface.co/distilbert/distilgpt2
